In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import bitsandbytes
from PIL import Image
from IPython.display import display
from pathlib import Path
import unicodedata
import sys
import torch
import open_clip
import chromadb
import re
import os
import numpy as np
import subprocess
from PIL import Image
from pocket_tts import TTSModel
import scipy.io.wavfile
from scipy.io import wavfile
import soundfile as sf
from datetime import timedelta

from sqlalchemy.orm import Session
from sqlalchemy import text

import boto3
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

In [4]:
root = Path.cwd()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))
from src.db.session import engine
from src.models import Pharaoh

In [5]:
!nvidia-smi

Sat Mar  7 17:28:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.07                 Driver Version: 566.07         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:64:00.0 Off |                  N/A |
| N/A   49C    P5              4W /   60W |     753MiB /   8188MiB |     27%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
with open(r"D:\1Graduation project\ECHO\experiments\video_generation\video_generation_pharaohs\Scripts\Ramesses II.txt", "r", encoding="utf-8") as file:
    text = file.read()

topics = [section.strip() for section in text.split("\n\n") if section.strip()]

for i, section in enumerate(topics, 1):
    print(f"Section {i}:")
    print(section)
    print("-" * 40)

Section 1:
Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war. At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side. He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.
----------------------------------------
Section 2:
Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region. This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns. Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.
----------------------------------------
Section 3:
He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength. Rameses faced a legendary battle against the 

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [8]:
import open_clip


model_path = "C:\\Users\\lidia\\Downloads\\open_clip_pytorch_model.bin"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained=model_path  
)
model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [12]:
'''device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)
model = model.to(device)
model.eval()'''

'device = "cuda" if torch.cuda.is_available() else "cpu"\n\nmodel, preprocess, tokenizer = open_clip.create_model_and_transforms(\n    "ViT-H-14",\n    pretrained="laion2b_s32b_b79k"\n)\nmodel = model.to(device)\nmodel.eval()'

In [13]:
script_path = r'D:\1Graduation project\ECHO\experiments\video_generation\video_generation_pharaohs\Scripts\Ramesses II.txt'
with open(script_path, 'r', encoding='utf-8') as file:
    script = file.read()
paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
paragraphs

['Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war. At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side. He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.',
 'Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region. This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns. Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.',
 "He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength. Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peace treaty between two great powers."

In [11]:
#Text to speech
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

Path("tts_Outputs").mkdir(exist_ok=True)

durations = []
images_needed = []
seconds = []

for i, p in enumerate(paragraphs):
    audio = tts_model.generate_audio(voice_state, p)
    output_path = f"tts_Outputs/output_{i}.wav"
    scipy.io.wavfile.write(output_path, tts_model.sample_rate, audio.numpy())

    Fs, data = wavfile.read(output_path)
    duration_seconds = len(data) / float(Fs)

    durations.append(duration_seconds)
    img_count = max(1, int(duration_seconds / 6))
    images_needed.append(img_count)

    section_seconds = duration_seconds / img_count
    seconds.extend([section_seconds] * img_count)
    
print(f"Paragraph {i+1}:")
print(f"Duration: {duration_seconds:.2f}s")
print(f"Images needed: {img_count}")
print(f"Seconds per image: {section_seconds:.2f}s\n")


Paragraph 4:
Duration: 22.80s
Images needed: 3
Seconds per image: 7.60s



In [14]:
i = 0
image_text_chunks = []
for p in paragraphs:
    sentences = re.split(r'(?<=[.!?]) +', p)
    sentences = [s.strip() for s in sentences if s.strip()]
    images_for_paragraph = images_needed[i]

    images_for_paragraph = min(images_for_paragraph, len(sentences))

    total_sentences = len(sentences)

    base = total_sentences // images_for_paragraph
    remainder = total_sentences % images_for_paragraph

    groups = []
    start = 0

    for img_idx in range(images_for_paragraph):
        extra = 1 if img_idx < remainder else 0
        end = start + base + extra

        chunk = " ".join(sentences[start:end])
        groups.append(chunk)

        start = end

    image_text_chunks.append(groups)
        
    i += 1
    print(f"Paragraph {i} split into {len(groups)} image text chunks.")
    print("-" * 40)
    print(groups)
print(image_text_chunks)

Paragraph 1 split into 3 image text chunks.
----------------------------------------
['Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war.', 'At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.', 'He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.']
Paragraph 2 split into 3 image text chunks.
----------------------------------------
['Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region.', 'This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns.', 'Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.']
Paragraph 3 split into 2 image text chunks.
----------------------------------------
["He continued his fat

In [15]:
# Merge all wav files
output_dir = Path("tts_Outputs")
wav_files = sorted([f for f in os.listdir("tts_Outputs") if f.endswith(".wav")])
audio_data = []
samplerate = None

for file_name in wav_files:
    file_path = os.path.join(output_dir, file_name)
    data, sr = sf.read(file_path)
    if samplerate is None:
        samplerate = sr
    audio_data.append(data)
    os.remove(file_path)

combined = np.concatenate(audio_data, axis=0)
sf.write("tts_Outputs/Ramesses_II.wav", combined, samplerate)

In [16]:
final_audio_path = output_dir / "Ramesses_II.wav"

print("Final merged audio saved:", final_audio_path)
print("Durations:", durations)
print("Images needed:", images_needed)
print("Seconds list length:", len(seconds))

Final merged audio saved: tts_Outputs\Ramesses_II.wav
Durations: [23.36, 19.84, 14.32, 22.8]
Images needed: [3, 3, 2, 3]
Seconds list length: 11


In [17]:
# FETCH IMAGES FROM DATABASE
from sqlalchemy.orm import Session
from sqlalchemy import text

name = Path(script_path).stem
total_images_required = sum(images_needed)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

fetched_images_ids = []
fetched_images_paths = []

with Session(engine) as session:
    result = session.execute(
        text("""
            SELECT 
                pi.id,
                pi.image_path,
                pi.image_embedding,
                pi.image_description
            FROM pharaohs_images pi
            JOIN pharaohs p ON pi.pharaoh_id = p.id
            WHERE p.name = :name
        """),
        {"name": name}
    )


    images_data = result.fetchall()

print(f"\nFound {len(images_data)} images for {name}")



Found 31 images for Ramesses II


In [18]:
#script embeddings
clip_tokenizer = open_clip.get_tokenizer("ViT-H-14")
script_chunk_embeddings = []

for paragraph_chunks in image_text_chunks:
    paragraph_embeddings = []
    
    for chunk in paragraph_chunks:
        script_tokens = clip_tokenizer([chunk]).to(device)
    
        with torch.no_grad():
            script_embedding = model.encode_text(script_tokens)
            script_embedding /= script_embedding.norm(dim=-1, keepdim=True)

        script_embedding = script_embedding.cpu().numpy()[0]
        paragraph_embeddings.append(script_embedding)
        print("Generated embedding for chunk:")
        print(chunk)
        print("-"*50)

    script_chunk_embeddings.append(paragraph_embeddings)

print("\nTotal paragraphs:", len(script_chunk_embeddings))      

Generated embedding for chunk:
Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war.
--------------------------------------------------
Generated embedding for chunk:
At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.
--------------------------------------------------
Generated embedding for chunk:
He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.
--------------------------------------------------
Generated embedding for chunk:
Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region.
--------------------------------------------------
Generated embedding for chunk:
This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns.
---------------------------------------

In [ ]:
import json

for paragraph_embeddings in script_chunk_embeddings:
    
    for script_embedding in paragraph_embeddings:
        ranked_images = []

        for row in images_data:
            image_id, image_path, image_embedding, image_description = row

            if image_embedding is None or image_description is None:
                continue
            
            if isinstance(image_embedding, str):
                image_embedding = json.loads(image_embedding)
                
            image_embedding = np.array(image_embedding)
            image_embedding = image_embedding / np.linalg.norm(image_embedding)

            tokens = clip_tokenizer([image_description]).to(device)

            with torch.no_grad():
                desc_embedding = model.encode_text(tokens)
                desc_embedding /= desc_embedding.norm(dim=-1, keepdim=True)

            desc_embedding = desc_embedding.cpu().numpy()[0]
            
            #calculate similarity
            image_sim = cosine_similarity(script_embedding, image_embedding)
            desc_sim = cosine_similarity(script_embedding, desc_embedding)
            
            final_similarity = (0.7 * image_sim) + (0.3 * desc_sim)
            ranked_images.append((final_similarity, image_id, image_path))
    
print(f"\nTotal ranked images: {len(ranked_images)}")

ranked_images.sort(reverse=True, key=lambda x: x[0])
print("\nTop 5 similarity scores:")
for sim, img_id, path in ranked_images[:5]:
    print(f"Image {img_id} → {sim:.4f}")


Total ranked images: 31

Top 5 similarity scores:
Image 649 → 0.2509
Image 660 → 0.2367
Image 657 → 0.2291
Image 650 → 0.2251
Image 656 → 0.2243


In [21]:
# SELECT UNIQUE IMAGES
for sim, image_id, image_path in ranked_images:

    if image_id not in fetched_images_ids:
        fetched_images_ids.append(image_id)
        fetched_images_paths.append(image_path)
        #print(f"Selected Image ID: {image_id} | similarity={sim:.4f}")

    if len(fetched_images_ids) >= total_images_required:
        break

print("\nFinal Selected Images:")
print(fetched_images_ids)


Final Selected Images:
[649, 660, 657, 650, 656, 666, 659, 665, 678, 676, 667]


In [22]:
load_dotenv()

ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY = os.getenv("R2_ACCESS_KEY")
SECRET_KEY = os.getenv("R2_SECRET_KEY")
BUCKET_NAME = os.getenv("R2_BUCKET_NAME")

session = boto3.session.Session()

client = session.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)

path = Path.joinpath(Path("data/video_generation/raw/pharaohs_images/"), name)
REMOTE_PREFIX = path

local_frames_dir = Path("temp_frames")
local_frames_dir.mkdir(exist_ok=True)

ordered_local_paths = []

def download_image(idx_key):
    idx, image_key = idx_key
    local_file = local_frames_dir / f"{idx:04d}.jpg"
    client.download_file(BUCKET_NAME, image_key, str(local_file))
    return str(local_file)

with ThreadPoolExecutor(max_workers=8) as executor:
    ordered_local_paths = list(
        executor.map(download_image, enumerate(fetched_images_paths))
    )

In [23]:
image_files = list(local_frames_dir.glob("*.jpg")) + list(local_frames_dir.glob("*.jpeg"))

In [24]:
# ---------- SETTINGS ----------
MAX_CHARS_PER_LINE = 42
MAX_LINES = 2
MIN_DURATION = 1.0  # minimum subtitle duration in seconds


# ---------- HELPERS ----------
def normalize_text(text):
    """
    Cleans problematic Unicode characters that break
    subtitle rendering in Pillow / MoviePy.
    """

    # First normalize Unicode composition
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "’": "'",
        "‘": "'",
        "‚": ",",
        "‛": "'",

        "“": '"',
        "”": '"',
        "„": '"',

        "—": "-",
        "–": "-",
        "―": "-",

        "…": "...",

        "\u00A0": " ",   # non-breaking space
        "\u200B": "",    # zero-width space
        "\u200C": "",
        "\u200D": "",
        "\uFEFF": "",    # BOM if embedded
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # Remove any remaining control characters
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C"
    )

    return text

def format_timestamp(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    millis = int((seconds - total_seconds) * 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60

    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


def split_into_sentences(text):
    return re.split(r'(?<=[.!?]) +', text.strip())


def split_long_text(text, max_chars=MAX_CHARS_PER_LINE):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        if len(current_line) + len(word) + 1 <= max_chars:
            current_line += (" " if current_line else "") + word
        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)

    # combine into blocks of max 2 lines
    blocks = []
    for i in range(0, len(lines), MAX_LINES):
        block = "\n".join(lines[i:i + MAX_LINES])
        blocks.append(block)

    return blocks


# ---------- MAIN FUNCTION ----------
def generate_srt(paragraphs, durations, output_path="subtitles.srt"):
    assert len(paragraphs) == len(durations), "Paragraphs and durations must match"

    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []

    for paragraph, duration in zip(paragraphs, durations):
        paragraph = normalize_text(paragraph)
        sentences = split_into_sentences(paragraph)

        # break into subtitle chunks
        chunks = []
        for sentence in sentences:
            chunks.extend(split_long_text(sentence))

        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)

        for chunk in chunks:
            chunk_char_count = len(chunk.replace("\n", ""))

            # proportional timing
            chunk_duration = max(
                MIN_DURATION,
                (chunk_char_count / total_chars) * duration
            )

            start_time = current_time
            end_time = current_time + chunk_duration

            srt_blocks.append(f"{subtitle_index}\n{format_timestamp(start_time)} --> {format_timestamp(end_time)}\n{chunk}\n\n")
            

            current_time = end_time
            subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print(f"SRT file saved to {output_path}")


generate_srt(paragraphs, durations, "tts_Outputs/output_subtitles.srt")

SRT file saved to tts_Outputs/output_subtitles.srt


In [25]:
def run_ffmpeg(cmd):
    cmd = [str(c) for c in cmd]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

def create_kenburns_clip(
    image_path,
    duration,
    output_path,
    fps=30,
    target_w=1920,
    target_h=1080
):
    total_frames = int(duration * fps)

    with Image.open(image_path) as img:
        img_w, img_h = img.size

    scale_factor = max(target_w / img_w, target_h / img_h)

    scaled_w = int(img_w * scale_factor)
    scaled_h = int(img_h * scale_factor)

    max_x = scaled_w - target_w
    max_y = scaled_h - target_h

    threshold = 40  # pixels — below this we zoom instead of pan

    # Smooth easing expression
    #ease = f"(0.5-0.5*cos(PI*t/{duration}))"
    ease = f"(0.5-0.5*cos(PI*n/{total_frames}))"
    
    # Decide motion
    if (img_w / img_h) > (target_w / target_h):
        # Wider image → horizontal movement possible
        if max_x > threshold:
            x_expr = f"floor({max_x}*{ease})"
            y_expr = f"{max_y}/2"
            zoom_filter = ""
        else:
            # Not enough horizontal space → subtle zoom
            zoom_filter = f",scale=iw*1.05:ih*1.05"
            x_expr = f"(iw-{target_w})/2"
            y_expr = f"(ih-{target_h})/2"
    else:
        # Taller image → vertical movement
        if max_y > threshold:
            x_expr = f"{max_x}/2"
            y_expr = f"floor({max_y}*{ease})"
            zoom_filter = ""
        else:
            # Not enough vertical space → subtle zoom
            zoom_filter = f",scale=iw*1.05:ih*1.05"
            x_expr = f"(iw-{target_w})/2"
            y_expr = f"(ih-{target_h})/2"

    vf = (
        f"scale={scaled_w}:{scaled_h}"
        f"{zoom_filter},"
        f"crop={target_w}:{target_h}:"
        f"x='{x_expr}':"
        f"y='{y_expr}'"
    )

    cmd = [
        "ffmpeg",
        "-y",
        "-loop", "1",
        "-framerate", str(fps),
        "-t", str(duration),
        "-i", image_path,
        "-vf", vf,
        "-frames:v", str(total_frames),
        "-c:v", "libx264",
        "-preset", "fast",
        "-pix_fmt", "yuv420p",
        "-vsync", "cfr",
        #"-video_track_timescale", "30",
        output_path
    ]

    run_ffmpeg(cmd)
    
    
'''    "-c:v", "h264_nvenc",
        "-preset", "p1",
        "-pix_fmt", "yuv420p",'''

'    "-c:v", "h264_nvenc",\n        "-preset", "p1",\n        "-pix_fmt", "yuv420p",'

In [26]:
def generate_all_clips(image_files, durations, temp_dir="temp_clips"):
    Path(temp_dir).mkdir(exist_ok=True)

    outputs = []

    for i, (img, dur) in enumerate(zip(image_files, durations)):
        out = f"{temp_dir}/clip_{i}.mp4"
        create_kenburns_clip(img, dur, out)
        outputs.append(out)

    return outputs

def concatenate_clips(clips, output_path):
    list_file = "tts_Outputs/concat_list.txt"

    with open(list_file, "w") as f:
        for clip in clips:
            f.write(f"file '{os.path.abspath(clip)}'\n")

    cmd = [
        "ffmpeg",
        "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", list_file,
        "-c:v", "libx264",
        "-preset", "fast",
        #"-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

In [ ]:
def add_audio(video_path, audio_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-i", audio_path,
        "-c:v", "copy",
        "-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

def add_subtitles(video_path, srt_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-vf", f"subtitles={srt_path}",
        "-c:v", "libx264",
        "-preset", "fast",
        "-c:a", "copy",
        output_path
    ]

    run_ffmpeg(cmd)

def cleanup_files():
    temp_dir = "temp_clips"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    temp_dir = "temp_frames"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    for f in os.listdir('tts_Outputs'):
        if f.startswith("combined") or f.startswith("with_audio") or f.startswith("concat_list") or f.startswith("output_subtitles") or f.endswith(".wav"):
            os.remove(os.path.join('tts_Outputs', f))

In [28]:
print(image_files)

[WindowsPath('temp_frames/0000.jpg'), WindowsPath('temp_frames/0001.jpg'), WindowsPath('temp_frames/0002.jpg'), WindowsPath('temp_frames/0003.jpg'), WindowsPath('temp_frames/0004.jpg'), WindowsPath('temp_frames/0005.jpg'), WindowsPath('temp_frames/0006.jpg'), WindowsPath('temp_frames/0007.jpg'), WindowsPath('temp_frames/0008.jpg'), WindowsPath('temp_frames/0009.jpg'), WindowsPath('temp_frames/0010.jpg')]


In [29]:
import subprocess

subprocess.run(["ffmpeg", "-version"], check=True)

CompletedProcess(args=['ffmpeg', '-version'], returncode=0)

In [30]:
# 1. Generate Ken Burns clips
clips = generate_all_clips(image_files, seconds)

# 2. Concatenate
concatenated = "tts_Outputs/combined.mp4"
concatenate_clips(clips, concatenated)

# 3. Add audio
with_audio = "tts_Outputs/with_audio.mp4"
add_audio(concatenated, "tts_Outputs/Ramesses_II.wav", with_audio)

generate_srt(paragraphs, durations, "tts_Outputs/output_subtitles.srt")
# 4. Add subtitles
final_output = "tts_Outputs/final_video.mp4"
add_subtitles(with_audio, "tts_Outputs/output_subtitles.srt", final_output)

cleanup_files()

print("Done:", final_output)

Running: ffmpeg -y -loop 1 -framerate 30 -t 7.786666666666666 -i temp_frames\0000.jpg -vf scale=1920:1263,crop=1920:1080:x='0/2':y='floor(183*(0.5-0.5*cos(PI*n/233)))' -frames:v 233 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_0.mp4
Running: ffmpeg -y -loop 1 -framerate 30 -t 7.786666666666666 -i temp_frames\0001.jpg -vf scale=1920:1433,crop=1920:1080:x='0/2':y='floor(353*(0.5-0.5*cos(PI*n/233)))' -frames:v 233 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_1.mp4
Running: ffmpeg -y -loop 1 -framerate 30 -t 7.786666666666666 -i temp_frames\0002.jpg -vf scale=1920:1272,crop=1920:1080:x='0/2':y='floor(192*(0.5-0.5*cos(PI*n/233)))' -frames:v 233 -c:v libx264 -preset fast -pix_fmt yuv420p -vsync cfr temp_clips/clip_2.mp4
Running: ffmpeg -y -loop 1 -framerate 30 -t 6.613333333333333 -i temp_frames\0003.jpg -vf scale=1920:1481,crop=1920:1080:x='0/2':y='floor(401*(0.5-0.5*cos(PI*n/198)))' -frames:v 198 -c:v libx264 -preset fast -pix_fmt yuv420p -